# Проект 1. Подорожание 1 марта 2026: что оно дало и чем мы за него заплатили

**Автор:** Рындин Даниил Александрович, группа БАЦРФ2601

**Бизнес-вопрос.** 1 марта 2026 года ПРАЙМ поднял цены: «Базовый» 199 -> 249 руб. (+25 %),
«Расширенный» 399 -> 449 руб. (+12,5 %). Сколько выручки это принесло, за счёт чего
именно, - и не заплатили ли мы за это клиентами: оттоком, притоком новых, уходом на
дешёвый тариф?

| Раздел | Что внутри |
|---|---|
| 1. Подготовка | подключение, окна сравнения |
| 2. Данные и зерно | что выгружаем и почему именно на таком уровне детализации |
| 3. Проверки данных | двойной счёт, дубли, дыра в апреле, справочник цен, август - до расчётов |
| 4. Гипотеза 0 | удерживает ли шеринг подписку (с неё всё началось) |
| 5. Гипотеза 1 | подорожание подняло выручку: факторное разложение «где» и «за счёт чего» |
| 6. Гипотезы 2-4 | отток, приток новых, уход на дешёвый тариф |
| 7. Сценарии | устойчив ли вывод: порядок подстановок, окна сравнения, сценарий «без подорожания», безубыточный отток |
| 8. Итог | сводка по гипотезам и вывод |

Ноутбук воспроизводим: все числа считаются из базы при запуске, ничего не вписано руками.
Перезапуск с нуля - **Kernel -> Restart Kernel and Run All Cells**, около 3-4 минут.

## 1. Подготовка

Сутки и месяцы считаем по UTC (`set time zone 'UTC'` внутри `q()`), как договорились на Л5.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

from prime.config import get_engine
from prime.factors import chain, contrib, waterfall

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


import matplotlib.dates as mdates

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12})
ACCENT, GREY, DARK = "#C1272D", "#9CA3AF", "#111827"
TARIFF = {1: "Базовый", 2: "Расширенный"}
RU_MONTH = {1: "янв", 2: "фев", 3: "мар", 4: "апр", 5: "май", 6: "июн",
            7: "июл", 8: "авг", 9: "сен", 10: "окт", 11: "ноя", 12: "дек"}


def ru(x: float, digits: int = 1) -> str:
    """Число с десятичной запятой, как принято в русском тексте."""
    return f"{x:.{digits}f}".replace(".", ",")


def month_axis(ax: plt.Axes) -> None:
    """Подписи оси времени: каждый третий месяц, по-русски."""
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda v, _: f"{RU_MONTH[mdates.num2date(v).month]} {mdates.num2date(v):%y}"))

print("подключились как:", q("select current_user as login").iloc[0, 0])

**Период и окна сравнения.**

- История в базе начинается 6 января 2025 года - январь 2025 неполный, поэтому берём с **февраля 2025**.
- **Август 2026 исключаем**: 22-23 августа разом закончились 27 тыс. подписок `partner_telecom`
  (разобрано на Л6), а 30 августа выгружено наполовину. Это другое событие, оно исказило бы и выручку,
  и отток (раздел 3.6).
- Разложение «за счёт чего» - **февраль -> март 2026**: последний месяц по старым ценам и первый по новым.
- Отток и приток сравниваем в **равных окнах по 5 месяцев**: октябрь 2025 - февраль 2026 («до»)
  против марта - июля 2026 («после»).

In [ ]:
FIRST, LAST = "2025-02-01", "2026-07-01"           # первый и последний месяц анализа
BASE_MONTH, PRICE_MONTH = "2026-02", "2026-03"     # последний месяц по старым ценам и первый по новым
BEFORE = [f"{m}" for m in pd.period_range("2025-10", "2026-02", freq="M").astype(str)]
AFTER = [f"{m}" for m in pd.period_range("2026-03", "2026-07", freq="M").astype(str)]

print("окно «до»:   ", BEFORE)
print("окно «после»:", AFTER)

## 2. Данные и зерно

**Считает база, а не ноутбук.** Сырых строк - миллионы, поэтому в pandas забираем уже свёрнутые
таблицы. Зерно каждой выбрано под вопрос, на который она отвечает:

| Таблица | Одна строка - это | Зачем такое зерно |
|---|---|---|
| `by_tariff` | месяц × тариф | выручка = активные подписки × платежей на подписку × цена; тариф - свойство **подписки**, поэтому у каждой строки есть своя активная база |
| `by_method` | месяц × способ оплаты | «где» по способам оплаты; способ оплаты - свойство **платежа**, а не подписки (одна подписка платит в разные месяцы по-разному - ДЗ-1, вопрос 2), поэтому активную базу на способ оплаты разложить нельзя и внутри способа раскладываем только «платежи × цена» |
| `days` | месяц | сколько дней с данными о платежах: месяцы разной длины, а в апреле дыра - сравниваем **в пересчёте на день** |
| `churn` | месяц × тариф | отток: подписки, активные **на начало** месяца, и сколько из них закончилось в месяце (определение Л5) |
| `new_subs` | месяц × тариф | приток: сколько подписок оформлено |

**Активная в месяце** подписка - началась до конца месяца и не закончилась до его начала (как на С6).
**Выручка** - только `status = 'success'`. Платёж соединяется с подпиской по ключу `subscription_id`
(«много платежей -> одна подписка», m:1): соединение не размножает строки. Таблицу участников
`subscription_members` к выручке **не присоединяем** - раздел 3.2 показывает почему.

In [ ]:
by_tariff = q("""
    with months as (
        select generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as ms
    ),
    act as (
        select to_char(ms, 'YYYY-MM') as month, s.tariff_id, count(*) as active
        from months
        join prime.subscriptions s
          on s.started_at < ms + interval '1 month' and (s.ended_at is null or s.ended_at >= ms)
        group by 1, 2
    ),
    rev as (
        select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id,
               count(*) as n_payments, sum(p.amount) as revenue
        from prime.payments p
        join prime.subscriptions s on s.subscription_id = p.subscription_id
        where p.status = 'success'
          and p.paid_at >= cast(:first as timestamptz)
          and p.paid_at < cast(:last as timestamptz) + interval '1 month'
        group by 1, 2
    )
    select a.month, a.tariff_id, a.active,
           coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
    from act a left join rev r using (month, tariff_id)
    order by 1, 2
""", first=FIRST, last=LAST)

by_method = q("""
    select to_char(paid_at, 'YYYY-MM') as month, payment_method,
           count(*) as n_payments, sum(amount) as revenue
    from prime.payments
    where status = 'success'
      and paid_at >= cast(:first as timestamptz)
      and paid_at < cast(:last as timestamptz) + interval '1 month'
    group by 1, 2
    order by 1, 2
""", first=FIRST, last=LAST)

# Дни, за которые в базе вообще есть платежи (любого статуса).
days = q("""
    select to_char(paid_at, 'YYYY-MM') as month, count(distinct paid_at::date) as days_with_data
    from prime.payments
    where paid_at >= cast(:first as timestamptz)
      and paid_at < cast(:last as timestamptz) + interval '1 month'
    group by 1
    order by 1
""", first=FIRST, last=LAST).set_index("month")["days_with_data"]

churn = q("""
    with months as (
        select generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as ms
    )
    select to_char(ms, 'YYYY-MM') as month, s.tariff_id,
           count(*) as active_start,
           count(*) filter (where s.ended_at < ms + interval '1 month') as ended,
           count(*) filter (where s.ended_at < ms + interval '1 month'
                              and s.cancel_reason = 'too_expensive') as too_expensive
    from months
    join prime.subscriptions s
      on s.started_at < ms and (s.ended_at is null or s.ended_at >= ms)
    group by 1, 2
    order by 1, 2
""", first=FIRST, last=LAST)

new_subs = q("""
    select to_char(started_at, 'YYYY-MM') as month, tariff_id, count(*) as new_subs
    from prime.subscriptions
    where started_at >= cast(:first as timestamptz)
      and started_at < cast(:last as timestamptz) + interval '1 month'
    group by 1, 2
    order by 1, 2
""", first=FIRST, last=LAST)

for name, frame in [("by_tariff", by_tariff), ("by_method", by_method), ("days", days),
                    ("churn", churn), ("new_subs", new_subs)]:
    print(f"{name:<10} {len(frame):>4} строк")

## 3. Проверки данных - до расчётов

Шесть ловушек, о которых говорили на лекциях. Каждую проверяем кодом и решаем, что с ней делать.

### 3.1 Что считается выручкой

`failed` - списание не прошло, `refunded` - деньги вернули. Выручка - только `success`.

In [ ]:
statuses = q("""
    select status, count(*) as attempts, sum(amount) as amount
    from prime.payments
    where paid_at >= '2026-02-01' and paid_at < '2026-04-01'
    group by 1 order by 2 desc
""")
statuses["share_pct"] = (statuses["attempts"] / statuses["attempts"].sum() * 100).round(2)
statuses

### 3.2 Двойной счёт из-за шеринга

Подпиской делятся, и участники лежат в `subscription_members`. Соединение платежей с участниками
одновременно **раздувает** выручку (платёж повторяется на каждого участника) и **теряет** её
(подписки без участников выпадают при `inner join`). Сверяем три числа за февраль-март 2026.

In [ ]:
control = q("""
    select
      (select sum(amount) from prime.payments
        where status = 'success' and paid_at >= '2026-02-01' and paid_at < '2026-04-01') as direct,
      (select sum(p.amount) from prime.payments p
         join prime.subscription_members m on m.subscription_id = p.subscription_id
        where p.status = 'success' and p.paid_at >= '2026-02-01' and p.paid_at < '2026-04-01') as naive_join
""").iloc[0]

ours = by_tariff.loc[by_tariff["month"].isin(["2026-02", "2026-03"]), "revenue"].sum()

print(f"напрямую из платежей:          {control['direct']:>15,.2f} руб.".replace(",", " "))
print(f"через участников (так нельзя): {control['naive_join']:>15,.2f} руб.  "
      f"x{control['naive_join'] / control['direct']:.2f}".replace(",", " "))
print(f"наша таблица by_tariff:        {ours:>15,.2f} руб.".replace(",", " "))

# Выручка в нашей таблице обязана совпасть с прямым счётом до копейки.
assert abs(ours - control["direct"]) < 0.01, "двойной счёт или потеря при соединении с подписками"

### 3.3 Двойные списания

Повтор той же суммы по той же подписке в пределах 20 минут - двойное списание (С3: ответ устойчив
от 15 минут до суток). Насколько они могут сдвинуть сравнение февраля и марта?

In [ ]:
dups = q("""
    select to_char(paid_at, 'YYYY-MM') as month,
           count(*) filter (where gap < interval '20 minutes') as duplicates,
           coalesce(sum(amount) filter (where gap < interval '20 minutes' and status = 'success'), 0) as dup_revenue,
           sum(amount) filter (where status = 'success') as revenue
    from (
        select paid_at, amount, status,
               paid_at - lag(paid_at) over (partition by subscription_id, amount order by paid_at) as gap
        from prime.payments
        where paid_at >= '2026-02-01' and paid_at < '2026-04-01'
    ) t
    group by 1 order by 1
""")
dups["dup_share_pct"] = (dups["dup_revenue"] / dups["revenue"] * 100).round(3)
dups

Доля двойных списаний в выручке - около десятой доли процента и почти одинаковая в обоих месяцах.
На разложение февраль -> март она не влияет, поэтому выручку не чистим, а оговариваем.

### 3.4 Дыра 14-15 апреля

В апреле платежи есть не за все дни. Если сравнивать месяцы целиком, апрель «просядет» на пустом месте.

In [ ]:
calendar_days = pd.Series(pd.PeriodIndex(days.index, freq="M").days_in_month, index=days.index)
gap = pd.DataFrame({"дней в месяце": calendar_days, "дней с платежами": days})
print(gap[gap["дней в месяце"] != gap["дней с платежами"]])

# А подписки в эти дни оформлялись и отменялись как обычно?
april = q("""
    select d::date as day,
           (select count(*) from prime.payments where paid_at::date = d::date) as payments,
           (select count(*) from prime.subscriptions where started_at::date = d::date) as started,
           (select count(*) from prime.subscriptions where ended_at::date = d::date) as ended
    from generate_series(cast('2026-04-13' as timestamptz), cast('2026-04-16' as timestamptz), interval '1 day') d
""")
april

Дыра только в **платежах**: оформления и отмены подписок за 14-15 апреля на месте. Поэтому
выручку и платежи делим на **дни с данными**, а отток и приток - на **календарные дни**.

### 3.5 Справочник цен

В `prime.tariffs` строка «Расширенный, 429 руб.» перекрывает соседние (С4). Какие цены списывались
на самом деле - смотрим в самих платежах.

In [ ]:
ref = q("select tariff_id, tariff_name, monthly_fee, valid_from, valid_to from prime.tariffs order by 1, 4")
pairs = ref.reset_index(names="row").merge(ref.reset_index(names="row"), on="tariff_id", suffixes=("_a", "_b"))
overlap = pairs[(pairs["row_a"] < pairs["row_b"])
                & (pairs["valid_from_a"] <= pairs["valid_to_b"]) & (pairs["valid_from_b"] <= pairs["valid_to_a"])]
print("пересекающихся интервалов в справочнике:", len(overlap))

charged = q("""
    select s.tariff_id, to_char(p.paid_at, 'YYYY-MM') as month,
           min(p.amount) as min_amount, max(p.amount) as max_amount, count(*) as n
    from prime.payments p join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= '2026-01-01' and p.paid_at < '2026-05-01'
    group by 1, 2 order by 1, 2
""")
charged

Списывались ровно 199/399 руб. до 1 марта и 249/449 руб. после; 429 руб. не списывалось ни разу.
**Цены берём из платежей, а не из справочника** - так ошибочная строка не может попасть в расчёт.

### 3.6 Август 2026

In [ ]:
aug = q("""
    select s.tariff_id,
           count(*) as active_start,
           count(*) filter (where s.ended_at < '2026-09-01') as ended,
           max(p_last.last_paid)::date as last_payment_day
    from prime.subscriptions s
    cross join (select max(paid_at) as last_paid from prime.payments) p_last
    where s.started_at < '2026-08-01' and (s.ended_at is null or s.ended_at >= '2026-08-01')
    group by 1 order by 1
""")
aug["churn_pct"] = (aug["ended"] / aug["active_start"] * 100).round(2)
aug

Отток в августе - больше 17 % против обычных ~4 %, а данные кончаются 30 августа. Это массовая
отмена `partner_telecom`, не связанная с ценой. **Окно анализа заканчиваем июлем 2026.**

### Сверка таблиц между собой

Выручка по тарифам и по способам оплаты - одни и те же платежи, сложенные по-разному.
Каждый месяц суммы обязаны совпасть.

In [ ]:
rev_t = by_tariff.groupby("month")["revenue"].sum()
rev_m = by_method.groupby("month")["revenue"].sum()
check = pd.DataFrame({"по тарифам": rev_t, "по способам оплаты": rev_m})
check["разница"] = check["по тарифам"] - check["по способам оплаты"]

assert check["разница"].abs().max() < 0.01, "таблицы не сходятся"
print("расхождение по месяцам, руб.: максимум", check["разница"].abs().max())
check.tail(6)

## 4. Гипотеза 0: шеринг удерживает подписку

С неё проект начинался. Логика бизнеса: если подписка общая, отменить её - значит «отключить» близких,
поэтому такие подписки должны жить дольше. Если так - шеринг надо продвигать.

**Как проверяем, чтобы не обмануть себя.** Сравнивать «подписками когда-либо делились» и «никогда»
нельзя: долгоживущая подписка просто успевает набрать участников, и шеринг «удержит» её задним числом.
Поэтому смотрим состояние **на начало каждого месяца** (зерно: подписка × месяц): были ли у подписки
участники в этот момент - и закончилась ли она в этом месяце. Отдельно по тарифам: у «Расширенного»
больше мест.

In [ ]:
subs = q("select subscription_id, tariff_id, started_at, ended_at, cancel_reason from prime.subscriptions")
members = q("select subscription_id, joined_at, left_at from prime.subscription_members")
print(f"подписок: {len(subs)}, строк участия: {len(members)}")

panel = []
for ms in pd.date_range("2025-03-01", LAST, freq="MS", tz="UTC"):
    me = ms + pd.offsets.MonthBegin(1)
    active = subs[(subs["started_at"] < ms) & (subs["ended_at"].isna() | (subs["ended_at"] >= ms))]
    live = members[(members["joined_at"] < ms) & (members["left_at"].isna() | (members["left_at"] >= ms))]
    n_live = live.groupby("subscription_id").size()           # участников на начало месяца
    part = active[["subscription_id", "tariff_id", "ended_at", "cancel_reason"]].copy()
    part["n_members"] = part["subscription_id"].map(n_live).fillna(0).astype(int)
    part["ended_in_month"] = part["ended_at"] < me
    part["month"] = ms.strftime("%Y-%m")
    panel.append(part)
panel = pd.concat(panel, ignore_index=True)
panel["shared"] = np.where(panel["n_members"] > 0, "делились", "не делились")

sharing = panel.groupby(["tariff_id", "shared"]).agg(sub_months=("ended_in_month", "size"),
                                                     ended=("ended_in_month", "sum"))
p = sharing["ended"] / sharing["sub_months"]
sharing["churn_pct"] = (p * 100).round(2)
sharing["se_pp"] = ((p * (1 - p) / sharing["sub_months"]) ** 0.5 * 100).round(3)   # стандартная ошибка
print(f"подписко-месяцев: {len(panel):,}".replace(",", " "))
sharing

In [ ]:
# Число участников: 0, 1, 2, ... (5 и больше - вместе).
by_members = (panel.assign(members=panel["n_members"].clip(upper=5))
              .groupby(["tariff_id", "members"])["ended_in_month"].mean().mul(100).round(2)
              .unstack("tariff_id").rename(columns=TARIFF))
print("отток в месяц, %, по числу участников на начало месяца")
print(by_members)

# Причины отмен у ушедших - одинаковы ли?
gone = panel[panel["ended_in_month"]]
(pd.crosstab(gone["cancel_reason"], gone["shared"], normalize="columns") * 100).round(1)

**Гипотеза 0 не подтвердилась.** Отток около 4 % в месяц и у подписок с участниками, и без них,
у обоих тарифов; от числа участников он тоже не зависит, и причины отмен совпадают. Разница между
группами меньше пары стандартных ошибок - это шум. Продвигать шеринг ради удержания данные
оснований не дают. Переходим к главному событию периода - подорожанию 1 марта.

## 5. Гипотеза 1: подорожание подняло выручку

### 5.1 Что случилось

Выручка в день - выручка месяца, делённая на дни с данными (раздел 3.4).

In [ ]:
m = by_tariff.groupby("month")[["active", "n_payments", "revenue"]].sum()
m["days"] = days
m["rev_day"] = m["revenue"] / m["days"]                          # выручка в день
m["per_sub_day"] = m["n_payments"] / m["days"] / m["active"]     # платежей в день на активную подписку
m["price"] = m["revenue"] / m["n_payments"]                      # цена одного платежа
m["growth_pct"] = m["rev_day"].pct_change() * 100

# Три множителя дают выручку в день точно.
assert ((m["active"] * m["per_sub_day"] * m["price"] - m["rev_day"]).abs() < 0.01).all()

m[["active", "n_payments", "revenue", "days", "rev_day", "price", "growth_pct"]].round(2)

In [ ]:
jump = m.loc[PRICE_MONTH, "growth_pct"]
usual_before = m.loc[BEFORE[1:], "growth_pct"].mean()
usual_after = m.loc[AFTER[1:], "growth_pct"].mean()

fig, ax = plt.subplots(figsize=(10, 4.4), layout="constrained")
x = pd.PeriodIndex(m.index, freq="M").to_timestamp()
ax.plot(x, m["rev_day"] / 1e6, color=GREY, marker="o", markersize=4, zorder=2)
i = list(m.index).index(PRICE_MONTH)
ax.plot(x[i - 1:i + 1], m["rev_day"].iloc[i - 1:i + 1] / 1e6, color=ACCENT, linewidth=2.5, zorder=3)
ax.scatter([x[i]], [m["rev_day"].iloc[i] / 1e6], color=ACCENT, zorder=4)
ax.annotate(f"1 марта: цены +25 % и +12,5 %\nвыручка в день +{ru(jump)} %",
            (x[i], m["rev_day"].iloc[i] / 1e6), xytext=(-170, 25), textcoords="offset points",
            color=ACCENT, arrowprops={"arrowstyle": "->", "color": ACCENT})
ax.set_title(f"Выручка в день выросла на {jump:.0f} % за один месяц - до и после рост "
             f"обычный: {ru(usual_before)} % и {ru(usual_after)} % в месяц", loc="left")
month_axis(ax)
ax.set_ylabel("млн руб. в день")
ax.set_ylim(0, m["rev_day"].max() / 1e6 * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.text(0, -0.16, "Выручка месяца / дни с данными о платежах (апрель - 28 дней из 30). "
        "Успешные платежи, сутки UTC. Февраль 2025 - июль 2026.",
        transform=ax.transAxes, fontsize=8, color="#6B7280")
plt.show()

### 5.2 За счёт чего: активные подписки × платежей на подписку × цена

Мультипликативная модель (Л6, слайд 13), в пересчёте на день:

> выручка в день = активные подписки × платежей в день на подписку × цена платежа

Цепные подстановки, порядок - **сначала «сколько», потом «почём»**: активные подписки -> платежей
на подписку -> цена. Сумма влияний обязана равняться изменению - это проверяет `assert` внутри `contrib`.

In [ ]:
FACTORS = ["active", "per_sub_day", "price"]
LABELS = {"active": "активные подписки", "per_sub_day": "платежей на подписку", "price": "цена платежа"}

x0 = m.loc[BASE_MONTH, FACTORS].to_dict()
x1 = m.loc[PRICE_MONTH, FACTORS].to_dict()
effects = pd.Series(contrib(x0, x1, FACTORS))
total_change = m.loc[PRICE_MONTH, "rev_day"] - m.loc[BASE_MONTH, "rev_day"]
assert abs(effects.sum() - total_change) < 0.01, "сумма влияний не равна изменению"

table = pd.DataFrame({"было (февраль)": x0, "стало (март)": x1}).T.round(5)
print(table.to_string())
print()
print((effects.rename(LABELS) / 1000).round(1).to_string(), " (тыс. руб. в день)")
print(f"изменение выручки в день: {total_change / 1000:+.1f} тыс. руб.; "
      f"цена - {effects['price'] / total_change:.0%} изменения")

In [ ]:
price_share = effects["price"] / total_change
ax = waterfall(effects.rename(LABELS) / 1000,
               title=f"{price_share * 100:.0f} % прироста выручки в марте дала цена, а не рост базы")
ax.set_xlabel("Февраль -> март 2026, выручка в день (на дни с данными о платежах).\n"
              "Порядок подстановок: активные подписки -> платежей на подписку -> цена.", fontsize=9)
plt.show()

### 5.3 Где: по тарифам - и эффект структуры

Сложение по тарифам, умножение внутри каждого (Л6, слайд 14). Внутри тарифа «цена платежа» -
это **чистая цена**. А на уровне всей выручки средняя цена платежа может меняться ещё и потому,
что сдвинулись доли тарифов, - это **эффект структуры** (Л6, слайд 15). Разница двух чисел его и показывает.

In [ ]:
t = by_tariff.copy()
t["days"] = t["month"].map(days)
t["rev_day"] = t["revenue"] / t["days"]
t["per_sub_day"] = t["n_payments"] / t["days"] / t["active"]
t["price"] = t["revenue"] / t["n_payments"]
t = t.set_index(["month", "tariff_id"])

rows = {}
for tariff_id in TARIFF:
    a = t.loc[(BASE_MONTH, tariff_id), FACTORS].to_dict()
    b = t.loc[(PRICE_MONTH, tariff_id), FACTORS].to_dict()
    rows[TARIFF[tariff_id]] = contrib(a, b, FACTORS)
inside = pd.DataFrame(rows).T
inside["итого"] = inside.sum(axis=1)
inside.loc["сумма тарифов"] = inside.sum()

# Сумма по тарифам равна изменению всей выручки в день.
assert abs(inside.loc["сумма тарифов", "итого"] - total_change) < 0.01

structure = effects["price"] - inside.loc["сумма тарифов", "price"]
print((inside.rename(columns=LABELS) / 1000).round(1).to_string(), "\n(тыс. руб. в день)")
print()
print(f"цена на уровне всей выручки:  {effects['price'] / 1000:+.1f} тыс. руб. в день")
print(f"чистая цена внутри тарифов:   {inside.loc['сумма тарифов', 'price'] / 1000:+.1f}")
print(f"эффект структуры тарифов:     {structure / 1000:+.1f}")

### 5.4 Где: по способам оплаты - и наш сегмент «кошелёк»

Способ оплаты - свойство платежа, поэтому внутри способа раскладываем двумя множителями (Л6, слайды 8 и 14):

> выручка способа в день = платежей в день × цена платежа

Сумма по способам равна изменению всей выручки - проверяем.

In [ ]:
pm = by_method.copy()
pm["days"] = pm["month"].map(days)
pm["pay_day"] = pm["n_payments"] / pm["days"]
pm["price"] = pm["revenue"] / pm["n_payments"]
pm = pm.set_index(["month", "payment_method"])

rows = []
for method in sorted(by_method["payment_method"].unique()):
    a, b = pm.loc[(BASE_MONTH, method)], pm.loc[(PRICE_MONTH, method)]
    by_count, by_price = chain(a["pay_day"], a["price"], b["pay_day"], b["price"])
    rows.append({"способ оплаты": method, "платежей в день": by_count, "цена платежа": by_price,
                 "итого": by_count + by_price,
                 "доля выручки в марте, %": b["revenue"] / pm.loc[PRICE_MONTH, "revenue"].sum() * 100})
by_pm = pd.DataFrame(rows).set_index("способ оплаты").sort_values("итого", ascending=False)
assert abs(by_pm["итого"].sum() - total_change) < 0.01, "способы оплаты не сходятся с итогом"

by_pm["цена, % итога способа"] = by_pm["цена платежа"] / by_pm["итого"] * 100
out = by_pm.copy()
out[["платежей в день", "цена платежа", "итого"]] /= 1000
print(out.round(1).to_string(), "\n(влияния - тыс. руб. в день)")

### 5.5 Все переходы сразу: цена «включилась» один раз

Если прирост марта - это цена, то влияние цены должно быть около нуля во всех остальных переходах.

In [ ]:
steps = {}
months = list(m.index)
for prev_month, month in zip(months[:-1], months[1:]):
    steps[month] = contrib(m.loc[prev_month, FACTORS].to_dict(), m.loc[month, FACTORS].to_dict(), FACTORS)
steps = pd.DataFrame(steps).T.rename(columns=LABELS)
steps["итого"] = steps.sum(axis=1)
assert (steps["итого"] - m["rev_day"].diff().dropna()).abs().max() < 0.01

(steps / 1000).round(1)      # тыс. руб. в день; строка - месяц, в который пришли

### 5.6 Сколько подорожание приносит каждый месяц

Сравнение с самим собой: те же успешные платежи марта-июля по старой цене (199/399 руб.) против
фактической. Это не прогноз, а арифметика по фактическим платежам - **при допущении, что эти
платежи прошли бы и по старой цене** (разделы 6-7 проверяют, нет ли признаков обратного).

In [ ]:
old_price = t.xs(BASE_MONTH, level="month")["price"]       # фактическая цена платежа в феврале
uplift = t.loc[AFTER].reset_index()
uplift["old_price"] = uplift["tariff_id"].map(old_price)
uplift["uplift"] = uplift["revenue"] - uplift["n_payments"] * uplift["old_price"]
uplift_m = uplift.pivot_table(index="month", columns="tariff_id", values="uplift", aggfunc="sum").rename(columns=TARIFF)
uplift_m["всего"] = uplift_m.sum(axis=1)
uplift_m["в день"] = uplift_m["всего"] / uplift_m.index.map(days)

print((uplift_m[["Базовый", "Расширенный", "всего"]] / 1e6).round(2).to_string(), "\n(млн руб.)")
print(f"за март-июль: {uplift_m['всего'].sum() / 1e6:.1f} млн руб.; "
      f"в среднем {uplift_m['в день'].mean() / 1e3:.0f} тыс. руб. в день")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2), layout="constrained")
labels = [f"{RU_MONTH[pd.Period(mm).month]} {pd.Period(mm).year}" for mm in uplift_m.index]
b1 = ax.bar(labels, uplift_m["Базовый"] / 1e6, color=ACCENT, label="Базовый: +50 руб. за платёж", zorder=2)
b2 = ax.bar(labels, uplift_m["Расширенный"] / 1e6, bottom=uplift_m["Базовый"] / 1e6,
            color="#E8A0A3", label="Расширенный: +50 руб. за платёж", zorder=2)
ax.bar_label(b2, labels=[f"{v / 1e6:.1f}".replace(".", ",") for v in uplift_m["всего"]], padding=3)
ax.set_title(f"Подорожание приносит {ru(uplift_m['всего'].mean() / 1e6)} млн руб. в месяц - "
             f"{uplift_m['всего'].sum() / 1e6:.0f} млн за пять месяцев", loc="left")
ax.set_ylabel("млн руб.")
ax.set_ylim(0, uplift_m["всего"].max() / 1e6 * 1.2)
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.text(0, -0.15, "Фактические успешные платежи × (новая цена - цена февраля 2026). "
        "Апрель ниже: нет данных за 14-15 апреля.", transform=ax.transAxes, fontsize=8, color="#6B7280")
plt.show()

## 6. Чем заплатили: гипотезы 2-4

Если цена отпугнула клиентов, след должен остаться в одном из трёх мест: больше отмен, меньше
новых подписок, переход на дешёвый тариф. Сравниваем **равные окна** по 5 месяцев до и после 1 марта.

### Гипотеза 2: подорожание вызвало отток

Отток месяца - доля подписок, активных на начало месяца, которые закончились в нём (Л5). Месяцы
разной длины: в феврале отменить успевают меньше. Поэтому пересчитываем на **30 дней**: отток × 30 /
дней в месяце.

In [ ]:
c = churn.copy()
c["days"] = pd.PeriodIndex(c["month"], freq="M").days_in_month
c["churn_30d_pct"] = c["ended"] / c["active_start"] * 30 / c["days"] * 100
c["period"] = np.where(c["month"].isin(BEFORE), "до", np.where(c["month"].isin(AFTER), "после", "-"))

window = c[c["period"] != "-"].assign(ended_30d=lambda d: d["ended"] * 30 / d["days"])
h2 = window.groupby(["tariff_id", "period"]).agg(active=("active_start", "sum"), ended_30d=("ended_30d", "sum"),
                                                 ended=("ended", "sum"), too_expensive=("too_expensive", "sum"))
p = h2["ended_30d"] / h2["active"]
h2["churn_30d_pct"] = (p * 100).round(2)
h2["se_pp"] = ((p * (1 - p) / h2["active"]) ** 0.5 * 100).round(3)
h2["too_expensive_share_pct"] = (h2["too_expensive"] / h2["ended"] * 100).round(1)
h2 = h2.rename(index=TARIFF, level="tariff_id")
h2[["active", "ended", "churn_30d_pct", "se_pp", "too_expensive_share_pct"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2), layout="constrained")
for tariff_id, color in ((1, ACCENT), (2, DARK)):
    s = c[c["tariff_id"] == tariff_id]
    xs = pd.PeriodIndex(s["month"], freq="M").to_timestamp()
    ax.plot(xs, s["churn_30d_pct"], color=color, marker="o", markersize=3)
    ax.annotate(f"{TARIFF[tariff_id]} ({'+25' if tariff_id == 1 else '+12,5'} % к цене)",
                (xs[-1], s["churn_30d_pct"].iloc[-1]), xytext=(6, -4 if tariff_id == 1 else 6),
                textcoords="offset points", color=color, fontsize=9)
price_day = pd.Timestamp("2026-03-01")
ax.axvline(price_day, color=GREY, linestyle="--", linewidth=1)
ax.text(price_day, 0.4, " 1 марта: подорожание", color="#6B7280", fontsize=9)
b, a = h2.loc[("Базовый", "до"), "churn_30d_pct"], h2.loc[("Базовый", "после"), "churn_30d_pct"]
ax.set_title(f"Отток после подорожания не вырос: у «Базового» {ru(b, 2)} % -> {ru(a, 2)} % "
             "в месяц, хотя он подорожал на 25 %", loc="left")
ax.set_ylabel("% подписок за 30 дней")
ax.set_ylim(0, c["churn_30d_pct"].max() * 1.3)
ax.set_xlim(right=pd.Timestamp("2026-09-20"))
month_axis(ax)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E5E7EB")
ax.text(0, -0.16, "Закончились в месяце / активные на начало месяца, пересчёт на 30 дней. "
        "Август 2026 (массовая отмена partner_telecom) исключён.", transform=ax.transAxes, fontsize=8, color="#6B7280")
plt.show()

### Гипотеза 3: подорожание отпугнуло новых клиентов

Приток - новые подписки в **календарный** день (оформления в дыру 14-15 апреля на месте, раздел 3.4).

### Гипотеза 4: клиенты ушли на дешёвый тариф

Если «Расширенный» стал невыгоден, его доля среди новых и среди активных подписок должна была упасть.

In [ ]:
ns = new_subs.pivot_table(index="month", columns="tariff_id", values="new_subs", aggfunc="sum").rename(columns=TARIFF)
ns["всего"] = ns.sum(axis=1)
ns["в день"] = ns["всего"] / pd.PeriodIndex(ns.index, freq="M").days_in_month
ns["доля Расширенного среди новых, %"] = ns["Расширенный"] / ns["всего"] * 100

act = by_tariff.pivot_table(index="month", columns="tariff_id", values="active", aggfunc="sum")
ns["доля Расширенного среди активных, %"] = act[2] / act.sum(axis=1) * 100

h34 = pd.DataFrame({
    "до": ns.loc[BEFORE, ["в день", "доля Расширенного среди новых, %", "доля Расширенного среди активных, %"]].mean(),
    "после": ns.loc[AFTER, ["в день", "доля Расширенного среди новых, %", "доля Расширенного среди активных, %"]].mean(),
})
h34["изменение"] = h34["после"] - h34["до"]
h34.round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
xs = pd.PeriodIndex(ns.index, freq="M").to_timestamp()

ax1.plot(xs, ns["в день"], color=DARK, marker="o", markersize=3)
ax1.axvline(pd.Timestamp("2026-03-01"), color=GREY, linestyle="--", linewidth=1)
ax1.set_title(f"Новых подписок в день: {h34.loc['в день', 'до']:.0f} до и "
              f"{h34.loc['в день', 'после']:.0f} после", loc="left")
ax1.set_ylabel("новых подписок в день")
ax1.set_ylim(0, ns["в день"].max() * 1.25)

ax2.plot(xs, ns["доля Расширенного среди активных, %"], color=DARK, marker="o", markersize=3, label="среди активных")
ax2.plot(xs, ns["доля Расширенного среди новых, %"], color=GREY, marker="o", markersize=3, label="среди новых")
ax2.axvline(pd.Timestamp("2026-03-01"), color=GREY, linestyle="--", linewidth=1)
ax2.set_title("Доля «Расширенного» не упала", loc="left")
ax2.set_ylabel("% подписок")
ax2.set_ylim(0, 60)
ax2.legend(frameon=False, loc="lower right")

for ax in (ax1, ax2):
    month_axis(ax)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#E5E7EB")
fig.suptitle("Подорожание не отпугнуло новых клиентов и не увело их на дешёвый тариф",
             x=0.01, ha="left", fontsize=12)
fig.text(0.01, -0.03, "Пунктир - 1 марта 2026. Новые подписки по started_at на календарный день; "
         "доля - по активным в месяце. Февраль 2025 - июль 2026.", fontsize=8, color="#6B7280")
plt.show()

## 7. Альтернативные сценарии: устойчив ли вывод

Вывод «прирост дала цена, а клиентами мы за это не заплатили» не должен зависеть от того,
как именно мы считали. Проверяем четыре вещи, которые могли бы его изменить.

### 7.1 Порядок подстановок

Совместный член достаётся фактору, который меняют вторым (Л6, слайды 10-11). Считаем все шесть
порядков и их среднее - оно от порядка не зависит (вектор Шепли, С6).

In [ ]:
import itertools
import math

orders = pd.DataFrame({
    " -> ".join(LABELS[f] for f in order): contrib(x0, x1, list(order))
    for order in itertools.permutations(FACTORS)
}).T.rename(columns=LABELS)
orders.loc["среднее по порядкам"] = orders.mean()
orders["доля цены, %"] = orders["цена платежа"] / total_change * 100

shown = orders.copy()
shown[list(LABELS.values())] /= 1000
print(shown.round(1).to_string(), "\n(влияния - тыс. руб. в день, февраль -> март 2026)")
price_share_orders = orders["доля цены, %"]

### 7.2 Окно сравнения

Февраль -> март - самое узкое окно. Проверяем шире: январь -> март и равные окна по 3 и по 5 месяцев
до и после 1 марта. Для окна из нескольких месяцев факторы считаем «на день окна»: выручка окна /
дни с данными, средняя активная база, цена - выручка / платежи.

In [ ]:
def pooled(months: list[str]) -> dict:
    """Три фактора выручки в день для окна из нескольких месяцев."""
    w = m.loc[months]
    days_sum = w["days"].sum()
    active = (w["active"] * w["days"]).sum() / days_sum           # средняя база, взвешенная по дням
    return {"active": active,
            "per_sub_day": w["n_payments"].sum() / days_sum / active,
            "price": w["revenue"].sum() / w["n_payments"].sum()}


WINDOWS = {
    "февраль -> март": (["2026-02"], ["2026-03"]),
    "январь -> март": (["2026-01"], ["2026-03"]),
    "3 мес.: дек-фев -> мар-май": (BEFORE[-3:], AFTER[:3]),
    "5 мес.: окт-фев -> мар-июл": (BEFORE, AFTER),
}
rows = {}
for name, (before_w, after_w) in WINDOWS.items():
    w0, w1 = pooled(before_w), pooled(after_w)
    eff = contrib(w0, w1, FACTORS)
    change = math.prod(w1.values()) - math.prod(w0.values())
    rows[name] = {**{LABELS[f]: v / 1000 for f, v in eff.items()},
                  "изменение": change / 1000, "доля цены, %": eff["price"] / change * 100}
windows = pd.DataFrame(rows).T
print(windows.round(1).to_string(), "\n(тыс. руб. в день)")

Чем шире окно, тем больше успевает вырасти база, и доля цены в изменении снижается - но цена
остаётся главным фактором в любом окне.

### 7.3 Сценарий «без подорожания»

Два независимых способа оценить, сколько дала цена в марте-июле:

1. **По платежам** (раздел 5.6): те же фактические платежи по старой цене.
2. **По тренду**: если бы выручка в день после февраля росла так же, как в ноябре-феврале,
   сколько бы её было - и насколько факт выше.

In [ ]:
g = m.loc[BEFORE[1:], "growth_pct"].mean() / 100                  # средний месячный рост до подорожания
steps_ahead = np.arange(1, len(AFTER) + 1)
scenario = pd.DataFrame({
    "факт": m.loc[AFTER, "rev_day"].values,
    "без подорожания, по тренду": m.loc[BASE_MONTH, "rev_day"] * (1 + g) ** steps_ahead,
}, index=AFTER)
scenario["разница с трендом"] = scenario["факт"] - scenario["без подорожания, по тренду"]
scenario["подорожание по платежам"] = uplift_m["в день"].values

print(f"рост в месяц до подорожания (ноя-фев): {g * 100:.2f} %")
print((scenario / 1000).round(1).to_string(), "\n(тыс. руб. в день)")
print(f"в среднем: по тренду +{scenario['разница с трендом'].mean() / 1000:.0f}, "
      f"по платежам +{scenario['подорожание по платежам'].mean() / 1000:.0f} тыс. руб. в день")

Оценка по тренду ниже: темп роста базы в процентах замедляется сам по себе (база растёт, а приток
новых постоянен - раздел 6), поэтому продлённый тренд завышает выручку «без подорожания». Оценка
по платежам точнее, но держится на допущении, что эти платежи прошли бы и по старой цене. Истина
между ними - и в обоих случаях речь о сотнях тысяч рублей в день.

### 7.4 Безубыточный отток: сколько клиентов мы могли себе позволить потерять

Грубо LTV подписки ≈ цена в месяц / месячный отток (Л5). Цена выросла в `r` раз - значит, LTV не
упадёт, пока отток вырос меньше чем в `r` раз. Считаем этот порог по каждому тарифу и сравниваем
с фактом.

In [ ]:
rows = []
for tariff_id, name in TARIFF.items():
    p_old = t.loc[(BASE_MONTH, tariff_id), "price"]
    p_new = t.loc[(PRICE_MONTH, tariff_id), "price"]
    c_before = h2.loc[(name, "до"), "churn_30d_pct"]
    c_after = h2.loc[(name, "после"), "churn_30d_pct"]
    rows.append({"тариф": name, "цена до": p_old, "цена после": p_new,
                 "рост цены, %": (p_new / p_old - 1) * 100,
                 "отток до, %": c_before,
                 "безубыточный отток, %": c_before * p_new / p_old,
                 "отток после, %": c_after})
breakeven = pd.DataFrame(rows).set_index("тариф")
breakeven["запас, п.п."] = breakeven["безубыточный отток, %"] - breakeven["отток после, %"]
breakeven.round(2)

Отток мог бы вырасти почти на процентный пункт у «Базового» и на полпункта у «Расширенного»,
и подорожание всё равно окупилось бы на горизонте жизни подписки. Фактически он не вырос вовсе.

## 8. Итог

### Сводка по гипотезам

In [ ]:
summary = pd.DataFrame([
    {"гипотеза": "0. Шеринг удерживает подписку",
     "что посчитали": f"отток с участниками / без: {sharing['churn_pct'].iloc[0]:.2f} % / {sharing['churn_pct'].iloc[1]:.2f} % (Базовый)",
     "итог": "не подтвердилась"},
    {"гипотеза": "1. Подорожание подняло выручку",
     "что посчитали": f"выручка в день {jump:+.1f} % в марте; цена - {price_share:.0%} прироста; "
                      f"+{uplift_m['всего'].mean() / 1e6:.1f} млн руб. в месяц",
     "итог": "подтвердилась"},
    {"гипотеза": "2. Подорожание вызвало отток",
     "что посчитали": f"Базовый {h2.loc[('Базовый', 'до'), 'churn_30d_pct']:.2f} -> {h2.loc[('Базовый', 'после'), 'churn_30d_pct']:.2f} %, "
                      f"Расширенный {h2.loc[('Расширенный', 'до'), 'churn_30d_pct']:.2f} -> {h2.loc[('Расширенный', 'после'), 'churn_30d_pct']:.2f} % в месяц",
     "итог": "не подтвердилась"},
    {"гипотеза": "3. Подорожание отпугнуло новых клиентов",
     "что посчитали": f"новых в день {h34.loc['в день', 'до']:.0f} -> {h34.loc['в день', 'после']:.0f}",
     "итог": "не подтвердилась"},
    {"гипотеза": "4. Клиенты ушли на дешёвый тариф",
     "что посчитали": f"доля Расширенного среди активных {h34.loc['доля Расширенного среди активных, %', 'до']:.1f} -> "
                      f"{h34.loc['доля Расширенного среди активных, %', 'после']:.1f} %",
     "итог": "не подтвердилась"},
    {"гипотеза": "Устойчивость вывода (раздел 7)",
     "что посчитали": f"доля цены {min(price_share_orders.min(), windows['доля цены, %'].min()):.0f}-"
                      f"{max(price_share_orders.max(), windows['доля цены, %'].max()):.0f} % при любом порядке и окне; "
                      f"запас по оттоку до безубыточности {breakeven['запас, п.п.'].min():.2f}-"
                      f"{breakeven['запас, п.п.'].max():.2f} п.п.",
     "итог": "вывод устойчив"},
])
pd.set_option("display.max_colwidth", 120)
summary

### Вывод за 30 секунд

_Заполняет автор: что случилось -> за счёт чего -> чего мы не знаем -> что делать (Л6, слайд 28).
Подробный текст - в `ЗАПИСКА-ПРОЕКТ-1.md`._

**Что случилось.** …

**За счёт чего.** …

**Чего мы не знаем.** …

**Что делать.** …